# 01 - Universes and data

Step 1 of `THESIS_GUIDE.md`. Before training or explaining anything, this notebook describes what the models will see in each of the five investment universes (DJI, NASDAQ-100, S&P 500, EURO STOXX 50, STOXX Europe 600):

1. **Model groups and universes**: the two summary tables of the thesis.
2. **Feature catalog**: the 157 Alpha158 features, the 360 Alpha360 columns and the market features, grouped by family, source and horizon (`xaifin.data.features`). All later comparisons of attributions use these groups.
3. **Membership and coverage**: how many stocks are in the index and have prices over time. FinBench trains on the constituents active at the start of each test year.
4. **Missing values** per feature family, over the study period.
5. **Data quality**: extreme labels, tickers mapped to the wrong company, and the filter that discards the samples with unusable prices (`xaifin.data.quality`).
6. **Labels**: the distribution of the forward return per year, i.e. the target and how noisy it is.
7. **Redundancy among Alpha158 features**: groups of near-duplicates. This matters for SHAP (credit is split among correlated features) and for counterfactuals (features cannot move independently).
8. **Market volatility** per year, the input for the period analysis of Step 8.

The feature files are large (0.7-27 GB), so `xaifin.data.summary.summarize()` reads each file once and caches the summaries in `code/results/data_summary/<universe>/`. Without a cache this takes 10-30 seconds for the core universes and about two minutes each for S&P 500 and STOXX Europe 600; with it, the notebook runs in seconds. The app page `1_Data` reads the same cache.

In [1]:
import pandas as pd
import plotly.io as pio
from IPython.display import display

from xaifin import viz
from xaifin.config import UNIVERSE_NAMES, UNIVERSES
from xaifin.data import quality
from xaifin.data.features import catalog
from xaifin.data.summary import (
    REDUNDANT_ABS_CORR,
    model_groups_table,
    name_mismatches,
    redundant_groups,
    redundant_pairs,
    summarize,
    universe_by_test_year,
    universe_table,
    volatility_by_year,
)

pio.renderers.default = "plotly_mimetype"  # renders in VS Code and JupyterLab
pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = 120

UNIVERSES = UNIVERSES  # all five; use config.CORE_UNIVERSES for DJI, NASDAQ-100 and EURO STOXX 50 only
REFRESH = False  # True recomputes the cached summaries (about 5 minutes for all five)

In [2]:
summaries = {u: summarize(u, refresh=REFRESH) for u in UNIVERSES}
pd.concat([summaries[u]["meta"] for u in UNIVERSES], ignore_index=True)

,universe,study_start,study_end,pred_len,rows_alpha158,rows_alpha360,correlation_sample_rows,seconds,created
0,dji,2015-01-01,2024-12-31,5,108465,108465,22174,7,2026-09-29T20:32:52
1,nasdaq100,2015-01-01,2024-12-31,5,440744,440744,87154,27,2026-09-29T20:33:19
2,sp500,2015-01-01,2024-12-31,5,1559563,1559563,315946,97,2026-09-29T20:41:56
3,sx5e,2015-01-01,2024-12-31,5,151086,151086,30342,10,2026-09-29T20:33:29
4,sxxp,2015-01-01,2024-12-31,5,1823474,1823474,351130,107,2026-09-29T20:43:43


## 1. Model groups and universes

The thesis compares explanations **within** a group (same features, same protocol) and, at the level of feature families, **across** the two groups. MASTER and MATCC also receive the market gate features of their region.

In [3]:
model_groups_table()

,group,models,input_per_stock,constant_features,market_gate_features,families
0,alpha158,"MASTER, MATCC, FactorVAE","[20, 157]",0,"63 US, 42 EU (MASTER, MATCC)","momentum, trend, position, volatility, candle, volume, price_volume"
1,alpha360,"HIST, DiscoverPLF, FinFormer","[1, 360]",2,none,"momentum, candle, volume"


One row per universe. *Index members* and *tickers with prices* are counted at the start of each test year (range over the test years; tickers include past members). *Stock-days* are the rows of the feature files in 2015-2024, and *discarded* the share of them removed by the data-quality filter (section 5). *Label IQR* is the interquartile range of the 5-day return after the filter, averaged over years: a robust measure of how noisy the target is. *Index volatility* is the annualized realized volatility of the universe's own index, averaged over 2015-2024.

In [4]:
universe_table(summaries).style.format({
    "stock_days": "{:,}", "discarded": "{:.2%}", "label_iqr": "{:.2%}", "share_positive": "{:.1%}", "index_volatility": "{:.1%}",
}).hide(axis="index")

universe,region,test_years,index_members,tickers_with_prices,stock_days,discarded,label_iqr,share_positive,index_volatility,most_volatile_year
DJI,US,2020-2024,30,43-44,"108,465",2.32%,3.89%,54.6%,14.3%,2020
NASDAQ-100,US,2020-2024,101-104,176-183,"440,744",0.34%,4.99%,53.9%,19.4%,2022
S&P 500,US,2020-2024,503-505,612-623,"1,559,563",0.20%,4.61%,54.1%,14.9%,2020
EURO STOXX 50,EU,2021-2024,50-51,60-61,"151,086",0.07%,4.19%,54.3%,16.8%,2020
STOXX Europe 600,EU,2021-2024,600-615,742-749,"1,823,474",0.90%,4.58%,53.2%,14.2%,2020


## 2. Feature catalog

Alpha158 and Alpha360 share almost no individual features, so attributions of the two model groups can only be compared on groups of features. Alpha360 values are prices of past days divided by today's close, so they are mostly **momentum**: the two sets overlap only on momentum, volume and candle features. **Source** and **horizon** are the fair dimensions for comparing the two groups. The Alpha features are the same in every universe; only the market features differ (US or EU indexes).

In [5]:
cat = catalog(UNIVERSES[0])
for dimension in ["family", "source", "horizon"]:
    display(pd.crosstab(cat[dimension], cat["group"], margins=True))

group,alpha158,alpha360,market,All
family,,,,
candle,11,4,0,15
market,0,0,63,63
momentum,35,296,0,331
position,45,0,0,45
price_volume,10,0,0,10
trend,20,0,0,20
volatility,11,0,0,11
volume,25,60,0,85
All,157,360,63,580


group,alpha158,alpha360,market,All
source,,,,
close,75,60,0,135
market,0,0,63,63
ohlc,42,240,0,282
price_volume,15,0,0,15
volume,25,60,0,85
All,157,360,63,580


group,alpha158,alpha360,market,All
horizon,,,,
long,58,240,24,322
medium,58,90,24,172
short,41,30,15,86
All,157,360,63,580


## 3. Index membership and coverage

*Index members* counts the constituents at each month end, from the membership history (`code/data/constituents/eodhd/`). *Tickers with prices* counts the stocks with a price on each day: it includes past members, so it is larger than the index. FinBench keeps only the members active at the start of each test year (table below). Tickers are counted as the maximum over the first five trading days, because European exchanges close on different holidays.

In [6]:
for u in UNIVERSES:
    s = summaries[u]
    viz.coverage_chart(s["membership"], s["coverage"], u).show()
    display(universe_by_test_year(s).set_index("test_year").T.rename_axis(UNIVERSE_NAMES[u]))

test_year,2020,2021,2022,2023,2024
DJI,,,,,
index_members,30,30,30,30,30
tickers_with_prices,44,43,43,43,43


test_year,2020,2021,2022,2023,2024
NASDAQ-100,,,,,
index_members,104,102,101,101,101
tickers_with_prices,178,182,183,176,178


test_year,2020,2021,2022,2023,2024
S&P 500,,,,,
index_members,504,505,505,503,503
tickers_with_prices,623,622,620,612,612


test_year,2020,2021,2022,2023,2024
EURO STOXX 50,,,,,
index_members,0,50,51,50,50
tickers_with_prices,60,60,61,61,61


test_year,2020,2021,2022,2023,2024
STOXX Europe 600,,,,,
index_members,0,614,615,600,600
tickers_with_prices,735,742,749,745,742


## 4. Missing values

Share of missing values per feature, averaged by family, over the study period (2015-2024). Rolling features are missing during the first days of a stock's history (up to 60 days for the longest windows). The models fill them with 0, the training median after the robust z-score.

In [7]:
for u in UNIVERSES:
    s = summaries[u]
    viz.missing_chart(s["missing"], cat, u).show()
    top = s["missing"].sort_values("missing_share", ascending=False).groupby("group").head(5)
    display(top.set_index(["group", "feature"]).rename_axis(index=[UNIVERSE_NAMES[u], "feature"]))

missing_share
DJI      feature                
alpha158 VMA60            0.0011
         SUMD60           0.0011
         CNTP60           0.0011
         CNTN60           0.0011
         SUMP60           0.0011
alpha360 LOW59            0.0005
         VWAP59           0.0005
         VOLUME59         0.0005
         CLOSE59          0.0005
         OPEN59           0.0005

missing_share
NASDAQ-100 feature                
alpha158   CORD60           0.0296
           CORD30           0.0256
           CORD20           0.0240
           CORD10           0.0221
           CORR60           0.0215
alpha360   HIGH59           0.0050
           OPEN59           0.0050
           CLOSE59          0.0050
           VWAP59           0.0050
           VOLUME59         0.0050

missing_share
S&P 500  feature                
alpha158 CORD60           0.0077
         CORD30           0.0058
         STD60            0.0055
         VSUMN60          0.0055
         VSUMP60          0.0055
alpha360 VOLUME59         0.0027
         LOW59            0.0027
         OPEN59           0.0027
         HIGH59           0.0027
         CLOSE59          0.0027

missing_share
EURO STOXX 50 feature                
alpha158      CORD60           0.0034
              VSUMN60          0.0031
              VSUMP60          0.0031
              SUMD60           0.0031
              VMA60            0.0031
alpha360      VWAP59           0.0016
              VOLUME59         0.0016
              LOW59            0.0016
              OPEN59           0.0016
              CLOSE59          0.0016

missing_share
STOXX Europe 600 feature                
alpha158         CORD60           0.0483
                 CORD30           0.0377
                 CORD20           0.0339
                 CORD10           0.0286
                 CORD5            0.0253
alpha360         HIGH59           0.0039
                 CLOSE59          0.0039
                 OPEN59           0.0039
                 VOLUME59         0.0039
                 LOW59            0.0039

## 5. Data quality

**Extreme labels.** Instruments whose price doubles or halves within 5 days somewhere in 2015-2024. Most are real moves: the crash of March 2020 alone produces dozens of −50% labels in S&P 500 and STOXX Europe 600. Others are data errors: bad or stale prices, a delisting recorded as a price of 0, a corporate action that was not adjusted. `affected_years` lists the test years whose rolling window contains such a label *and* whose universe contains the instrument: only these reach a model. The table keeps the likely errors, a price that doubles or loses three quarters in 5 days; the full list is in the app (`1_Data`, tab *Labels*). `discarded_days` counts the labels removed by the filter below.

**Ticker names.** Constituents whose name in the membership file differs from the company of the listing the prices come from (`<universe>_info.csv`). Most rows are abbreviations or renames (IAG, GSK, HP Inc); a few are tickers mapped to the wrong company, whose prices and features belong to another firm. The *Observations* list the confirmed ones.

In [8]:
pct = {"min_label": "{:.0%}", "max_label": "{:.0%}", "min_price": "{:.4g}", "first": "{:%Y-%m-%d}", "last": "{:%Y-%m-%d}"}
cols = ["instrument", "name", "days", "first", "last", "min_label", "max_label", "min_price", "affected_years", "discarded_days"]
for u in UNIVERSES:
    extreme = summaries[u]["extreme_labels"]
    affected = extreme[extreme["affected_years"] != ""]
    likely_errors = affected[(affected["max_label"] >= 1.0) | (affected["min_label"] <= -0.75)]
    names = name_mismatches(u)
    print(f"{UNIVERSE_NAMES[u]}: {len(extreme)} instruments with an extreme label, {len(affected)} reaching a model, "
          f"{len(likely_errors)} likely errors; {len(names)} ticker names to check")
    if len(likely_errors):
        display(likely_errors[cols].style.format(pct).hide(axis="index").set_caption(f"{UNIVERSE_NAMES[u]}: likely errors"))
    if len(names):
        display(names.style.format({"similarity": "{:.2f}"}).hide(axis="index").set_caption(f"{UNIVERSE_NAMES[u]}: ticker names"))

DJI: 1 instruments with an extreme label, 1 reaching a model, 0 likely errors; 3 ticker names to check


ticker,name,listing_name,similarity,test_years
PRG.US,The Procter & Gamble Company,PROG Holdings Inc,0.18,"2020, 2021, 2022, 2023, 2024"
HPQ.US,Hewlett-Packard Company,HP Inc,0.21,
GE.US,General Electric Company,GE Aerospace,0.39,


NASDAQ-100: 12 instruments with an extreme label, 4 reaching a model, 1 likely errors; 4 ticker names to check


instrument,name,days,first,last,min_label,max_label,min_price,affected_years,discarded_days
RIVN.US,Rivian Automotive,1,2021-11-09,2021-11-09,121%,121%,8.4,2023,0


ticker,name,listing_name,similarity,test_years
VBVT.US,Viacom,Viabuilt Ventures Inc,0.30,
ON.US,Onsemi,ON Semiconductor Corporation,0.35,2024
GOLD.US,Randgold Resources,"Gold.com, Inc.",0.38,
LIFE_old.US,Life Technologies,Lifeline Systems Inc,0.43,


...

EURO STOXX 50: 2 instruments with an extreme label, 0 reaching a model, 0 likely errors; 2 ticker names to check


ticker,name,listing_name,similarity,test_years
BMW.XETRA,BMW,Bayerische Motoren Werke Aktiengesellschaft,0.13,"2021, 2022, 2023, 2024"
UMG.AS,UMG,Universal Music Group N.V.,0.21,


STOXX Europe 600: 41 instruments with an extreme label, 30 reaching a model, 12 likely errors; 19 ticker names to check


instrument,name,days,first,last,min_label,max_label,min_price,affected_years,discarded_days
ORP.PA,ORPEA,17,2022-01-20,2023-11-29,-94%,855%,7.9,2022,11
BEIJ-B.ST,BEIJER REF AB,4,2019-05-31,2019-08-05,-89%,809%,6.238,"2021, 2022, 2023, 2024",4
VAR1.STU,VARTA AG,20,2018-06-21,2024-10-07,-87%,327%,1.382,2021,19
0RH1.LSE,KINNEVIK B,2,2024-11-28,2024-12-05,-71%,266%,54.91,2024,2
FCHRF.US,GEORG FISCHER,5,2022-11-16,2022-11-22,251%,251%,11.31,2022,5
ORRON.ST,LUNDIN ENERGY,50,2019-03-25,2022-03-31,100%,181%,0.0002,"2021, 2022",20
POLY.LSE,POLYMETAL INTERNATIONAL,14,2022-02-21,2023-08-14,-99%,169%,2.15,2022,5
UNPRF.US,UNIPER,3,2022-05-31,2022-11-08,-58%,127%,39.59,2022,0
ARGX.BR,ARGENX,1,2017-12-07,2017-12-07,116%,116%,7.4,"2021, 2022",0
HBR.LSE,HARBOUR ENERGY,6,2016-01-05,2020-03-06,-81%,-56%,167.2,"2023, 2024",0


ticker,name,listing_name,similarity,test_years
SSE.LSE,SCOTTISH & SOUTHERN ENERGY,SSE PLC,0.12,"2021, 2022, 2023, 2024"
IAG.MC,IAG,International Consolidated Airlines Group S.A,0.12,"2021, 2022, 2023, 2024"
BMW.XETRA,BMW,Bayerische Motoren Werke Aktiengesellschaft,0.13,"2021, 2022, 2023, 2024"
PZU.WAR,PZU GROUP,Powszechny Zaklad Ubezpieczen SA,0.20,"2021, 2022, 2023, 2024"
RDC.XETRA,SHOP APOTHEKE EUROPE,Redcare Pharmacy NV,0.21,
WDP.BR,WDP,Warehouses de Pauw Comm VA,0.21,"2021, 2022, 2023, 2024"
EDF.PA,EDF,Electricite de France S.A.,0.21,"2021, 2022, 2023"
UMG.AS,UMG,Universal Music Group N.V.,0.21,"2022, 2023, 2024"
SDF.XETRA,K + S,K+S Aktiengesellschaft,0.22,"2023, 2024"
DOCM.SW,ZUR ROSE GROUP,DocMorris AG,0.23,"2021, 2022"


### The data-quality filter

The thesis discards the samples whose prices cannot be trusted (decided 2026-09-29; FinBench keeps them). A sample, stock *i* on day *t*, is discarded when one of these problems falls in the days it reads, from the first price behind its input features to the end of its label (`xaifin/data/quality.py`):

- **wrong company**: the three tickers of the table above whose prices belong to another firm (DJI `PRG.US`, STOXX Europe 600 `TEN.US` and `TOM.F`);
- **stale price**: the same adjusted close for 20 or more consecutive trading days (a delisted or unlisted ticker whose last price is repeated);
- **price below 0.01**: an error or a quote after a delisting;
- **price jump**: a price that triples, or falls to a third, from one day to the next. No index member moves like this; the cases in these files are bad prices, unadjusted corporate actions and quotes after a bank seizure. The only real crash they include is Indivior's −72% of 2019-04-10.

The filter removes samples, not rows of the files, so input sequences never join the days before and after a gap. It applies to training, validation and test alike (`config.CLEAN_DATA`; turned off only to reproduce FinBench in Step 2). The counts below use the Alpha158 input (20 days of features, the longest the models read).

In [9]:
for u in UNIVERSES:
    d = summaries[u]["discarded"]
    rows = summaries[u]["meta"]["rows_alpha158"].iloc[0]
    by_rule = d.groupby("reason")["samples"].sum().reindex(quality.REASONS, fill_value=0)
    print(f"{UNIVERSE_NAMES[u]}: {d['samples'].sum():,} of {rows:,} samples discarded ({d['samples'].sum() / rows:.2%}); "
          + ", ".join(f"{rule} {n:,}" for rule, n in by_rule.items() if n))
    if len(d):
        display(d.head(10).style.format({"first": "{:%Y-%m-%d}", "last": "{:%Y-%m-%d}"}).hide(axis="index").set_caption(UNIVERSE_NAMES[u]))

DJI: 2,516 of 108,465 samples discarded (2.32%); wrong_company 2,516


instrument,name,reason,samples,first,last
PRG.US,The Procter & Gamble Company,wrong_company,2516,2015-01-02,2024-12-31


NASDAQ-100: 1,488 of 440,744 samples discarded (0.34%); stale_price 1,243, min_price 6, price_jump 239


instrument,name,reason,samples,first,last
VBVT.US,Viacom,stale_price,1243,2020-01-28,2024-12-31
VBVT.US,Viacom,price_jump,239,2016-08-17,2020-01-27
DISH.US,Dish Network,min_price,6,2023-12-28,2024-01-25


S&P 500: 3,128 of 1,559,563 samples discarded (0.20%); stale_price 2,695, min_price 131, price_jump 302


instrument,name,reason,samples,first,last
MON.US,Monument Circle Acquisition Corp,stale_price,782,2018-05-30,2021-07-07
AMCR.US,Amcor PLC,stale_price,739,2015-01-02,2018-03-13
XL.US,XL Fleet Corp,stale_price,659,2018-09-04,2021-04-16
SW.US,Smurfit WestRock plc,stale_price,274,2015-06-03,2017-03-09
CHK.US,Chesapeake Energy Corporation,stale_price,241,2020-06-19,2021-06-03
SBNY.US,Signature Bank,min_price,110,2023-10-12,2024-03-20
SBNY.US,Signature Bank,price_jump,92,2023-03-07,2024-04-01
KDP.US,Keurig Dr Pepper Inc,price_jump,85,2018-07-02,2018-10-30
INFO_old1.US,Ihs Markit Ltd,price_jump,70,2015-01-02,2015-04-14
FTR.US,Frontier Communications Corporation,price_jump,33,2020-03-13,2020-04-29


EURO STOXX 50: 106 of 151,086 samples discarded (0.07%); price_jump 106


instrument,name,reason,samples,first,last
VIV.PA,VIVENDI,price_jump,106,2015-12-23,2024-12-31


STOXX Europe 600: 16,426 of 1,823,474 samples discarded (0.90%); wrong_company 5,058, stale_price 10,231, min_price 254, price_jump 883


instrument,name,reason,samples,first,last
TOM.F,TOMRA SYSTEMS,wrong_company,2542,2015-01-02,2024-12-30
AKBLF.US,ALK-ABELLO B,stale_price,2518,2015-01-02,2024-12-31
TEN.US,TENARIS,wrong_company,2516,2015-01-02,2024-12-31
FCHRF.US,GEORG FISCHER,stale_price,2516,2015-01-02,2024-12-31
ORRON.ST,LUNDIN ENERGY,stale_price,1144,2015-01-02,2019-07-19
VAR1.STU,VARTA AG,stale_price,967,2015-01-02,2018-10-17
0ROY.LSE,DAVIDE CAMPARI,stale_price,923,2021-05-06,2024-12-31
EVR.LSE,EVRAZ,stale_price,705,2022-03-15,2024-12-31
SAABY.US,SAAB B,stale_price,502,2020-10-16,2024-09-09
MPI0.F,BANCA MONTE DEI PASCHI,stale_price,298,2016-12-16,2018-02-15


Standard deviation of the 5-day label per year, before and after the filter. The bad prices inflate it by up to twenty times (EURO STOXX 50 in 2015, STOXX Europe 600 in 2024); after the filter it follows market volatility.

In [10]:
label_std = pd.concat({
    (UNIVERSE_NAMES[u], version): summaries[u][table].set_index("year")["std"]
    for u in UNIVERSES for version, table in [("before", "label_stats"), ("after", "label_stats_clean")]
}, axis=1)
label_std.style.format("{:.1%}")

## 6. Labels

The target is FinBench's forward return over the horizon $L$ (default 5 days): $r_{i,t} = (p_{i,t+L} - p_{i,t}) / p_{i,t}$ on the adjusted close, shown **before** the daily cross-sectional z-score the models apply. It covers every ticker with prices, not only the test universes, after the data-quality filter of section 5. The spread of the boxes across years is market volatility: the z-score removes it from the target, but not from the features.

In [11]:
for u in UNIVERSES:
    s = summaries[u]
    viz.label_chart(s["label_stats_clean"], u).show()
    cols = ["year", "n", "mean", "std", "p05", "median", "p95", "share_positive"]
    display(s["label_stats_clean"][cols].set_index("year").rename_axis(UNIVERSE_NAMES[u]))

,n,mean,std,p05,median,p95,share_positive
DJI,,,,,,,
2015,10584.0000,0.0009,0.0339,-0.0516,0.0013,0.0517,0.5186
2016,10584.0000,0.0045,0.0324,-0.0422,0.0032,0.0539,0.5591
2017,10542.0000,0.0050,0.0252,-0.0313,0.0041,0.0439,0.5881
2018,10542.0000,-0.0007,0.0369,-0.0657,0.0019,0.0529,0.5278
2019,10783.0000,0.0046,0.0339,-0.0492,0.0056,0.0542,0.5927
2020,10699.0000,0.0034,0.0650,-0.0949,0.0046,0.0985,0.5475
2021,10584.0000,0.0049,0.0358,-0.0492,0.0035,0.0619,0.5535
2022,10542.0000,-0.0011,0.0478,-0.0793,-0.0002,0.0746,0.4969
2023,10500.0000,0.0032,0.0360,-0.0519,0.0025,0.0594,0.5339


,n,mean,std,p05,median,p95,share_positive
NASDAQ-100,,,,,,,
2015,43091.0000,0.0011,0.0454,-0.0679,0.0014,0.0659,0.5177
2016,42831.0000,0.0036,0.0500,-0.0666,0.0033,0.0732,0.5449
2017,42457.0000,0.0051,0.0395,-0.0513,0.0047,0.0614,0.5771
2018,43150.0000,0.0008,0.0515,-0.0798,0.0020,0.0755,0.5220
2019,44180.0000,0.0066,0.0467,-0.0614,0.0061,0.0759,0.5790
2020,44954.0000,0.0091,0.0759,-0.1019,0.0076,0.1223,0.5680
2021,45668.0000,0.0042,0.0565,-0.0776,0.0035,0.0853,0.5412
2022,44601.0000,-0.0036,0.0702,-0.1136,-0.0024,0.1027,0.4787
2023,43833.0000,0.0053,0.0528,-0.0698,0.0034,0.0865,0.5406


,n,mean,std,p05,median,p95,share_positive
S&P 500,,,,,,,
2015,154321.0000,0.0001,0.0400,-0.0628,0.0010,0.0574,0.5132
2016,156641.0000,0.0043,0.0432,-0.0597,0.0038,0.0686,0.5533
2017,157699.0000,0.0043,0.0339,-0.0447,0.0042,0.0526,0.5734
2018,157089.0000,-0.0009,0.0442,-0.0729,0.0013,0.0621,0.5150
2019,157038.0000,0.0053,0.0432,-0.0574,0.0057,0.0658,0.5803
2020,156493.0000,0.0056,0.0842,-0.1151,0.0053,0.1254,0.5475
2021,155747.0000,0.0055,0.0455,-0.0630,0.0048,0.0754,0.5581
2022,154976.0000,-0.0013,0.0573,-0.0916,-0.0001,0.0846,0.4974
2023,152536.0000,0.0029,0.0467,-0.0647,0.0028,0.0722,0.5330


,n,mean,std,p05,median,p95,share_positive
EURO STOXX 50,,,,,,,
2015,14464.0000,0.0030,0.0393,-0.0598,0.0035,0.0659,0.5371
2016,14633.0000,0.0034,0.0409,-0.0626,0.0042,0.0683,0.5512
2017,14632.0000,0.0034,0.0255,-0.0350,0.0028,0.0441,0.5503
2018,14722.0000,-0.0012,0.0337,-0.0565,0.0000,0.0496,0.4978
2019,15021.0000,0.0057,0.0312,-0.0442,0.0054,0.0552,0.5817
2020,15336.0000,0.0033,0.0731,-0.0960,0.0036,0.0989,0.5357
2021,15453.0000,0.0046,0.0351,-0.0500,0.0043,0.0591,0.5598
2022,15651.0000,-0.0006,0.0467,-0.0766,0.0009,0.0720,0.5094
2023,15533.0000,0.0038,0.0350,-0.0491,0.0044,0.0549,0.5655


,n,mean,std,p05,median,p95,share_positive
STOXX Europe 600,,,,,,,
2015,167500.0000,0.0032,0.0429,-0.0616,0.0029,0.0686,0.5320
2016,172918.0000,0.0038,0.0456,-0.0658,0.0037,0.0733,0.5437
2017,175023.0000,0.0047,0.0327,-0.0418,0.0037,0.0537,0.5598
2018,178134.0000,-0.0013,0.0410,-0.0655,0.0000,0.0589,0.4932
2019,181636.0000,0.0059,0.0393,-0.0531,0.0050,0.0670,0.5652
2020,186194.0000,0.0040,0.0724,-0.1026,0.0041,0.1093,0.5368
2021,187789.0000,0.0041,0.0431,-0.0589,0.0037,0.0667,0.5468
2022,187750.0000,-0.0017,0.0580,-0.0925,-0.0002,0.0841,0.4949
2023,185271.0000,0.0023,0.0455,-0.0643,0.0022,0.0699,0.5255


## 7. Redundancy among Alpha158 features

Absolute Spearman correlation between the 157 Alpha158 features, pooled over every Wednesday of the study period, in dendrogram order so that correlated features sit together. Features with $|\rho| \geq 0.9$ with every other member of their group are near-duplicates (complete linkage). For explanations this means:
- **SHAP / permutation importance** split or blur credit within a group, so importance should also be reported per group, not only per feature;
- **counterfactuals** (Step 9) must move a group together, since its features cannot change independently.

In [12]:
for u in UNIVERSES:
    s = summaries[u]
    viz.correlation_chart(s["alpha158_corr"], s["alpha158_clusters"], u).show()
    groups = redundant_groups(s["alpha158_clusters"])
    n_clusters = s["alpha158_clusters"]["cluster"].nunique()
    print(f"{UNIVERSE_NAMES[u]}: {n_clusters} groups for 157 features; {groups['size'].sum()} features "
          f"sit in {len(groups)} groups of near-duplicates; {len(redundant_pairs(s['alpha158_corr']))} pairs with |rho| >= {REDUNDANT_ABS_CORR}")
    display(groups.head(10))

DJI: 104 groups for 157 features; 82 features sit in 29 groups of near-duplicates; 115 pairs with |rho| >= 0.9

,size,features
0,4,"ROC5, SUMN5, SUMP5, SUMD5"
1,4,"ROC10, SUMN10, SUMP10, SUMD10"
2,4,"ROC30, SUMN30, SUMP30, SUMD30"
3,4,"ROC20, SUMN20, SUMP20, SUMD20"
4,4,"ROC60, SUMD60, SUMP60, SUMN60"
5,3,"CNTN10, CNTP10, CNTD10"
6,3,"QTLU60, RANK60, RSV60"
7,3,"CNTN20, CNTP20, CNTD20"
8,3,"CNTP30, CNTN30, CNTD30"
9,3,"CNTN60, CNTP60, CNTD60"


NASDAQ-100: 105 groups for 157 features; 80 features sit in 28 groups of near-duplicates; 110 pairs with |rho| >= 0.9


,size,features
0,4,"ROC5, SUMN5, SUMP5, SUMD5"
1,4,"ROC10, SUMN10, SUMP10, SUMD10"
2,4,"ROC30, SUMN30, SUMP30, SUMD30"
3,4,"ROC60, SUMN60, SUMP60, SUMD60"
4,4,"ROC20, SUMP20, SUMN20, SUMD20"
5,4,"RANK60, RSV60, MA60, QTLU60"
6,3,"CNTN20, CNTP20, CNTD20"
7,3,"CNTN30, CNTP30, CNTD30"
8,3,"RSV20, RANK20, RANK30"
9,3,"CNTN60, CNTP60, CNTD60"


S&P 500: 103 groups for 157 features; 81 features sit in 27 groups of near-duplicates; 114 pairs with |rho| >= 0.9

,size,features
0,4,"ROC5, SUMN5, SUMP5, SUMD5"
1,4,"QTLU20, RSV20, RANK20, RANK30"
2,4,"ROC10, SUMN10, SUMP10, SUMD10"
3,4,"ROC20, SUMN20, SUMP20, SUMD20"
4,4,"RANK60, RSV60, MA60, QTLU60"
5,4,"ROC30, SUMN30, SUMP30, SUMD30"
6,4,"ROC60, SUMN60, SUMP60, SUMD60"
7,3,"CNTP10, CNTN10, CNTD10"
8,3,"QTLU10, RANK10, RSV10"
9,3,"RSV30, MA30, QTLU30"


EURO STOXX 50: 103 groups for 157 features; 82 features sit in 28 groups of near-duplicates; 119 pairs with |rho| >= 0.9


,size,features
0,4,"QTLU20, RSV20, RANK20, RANK30"
1,4,"ROC5, SUMN5, SUMP5, SUMD5"
2,4,"ROC60, SUMP60, SUMN60, SUMD60"
3,4,"ROC10, SUMN10, SUMP10, SUMD10"
4,4,"ROC20, SUMN20, SUMP20, SUMD20"
5,4,"ROC30, SUMN30, SUMP30, SUMD30"
6,3,"RSV10, QTLU10, RANK10"
7,3,"CNTP30, CNTN30, CNTD30"
8,3,"RSV30, MA30, QTLU30"
9,3,"CNTP10, CNTN10, CNTD10"


STOXX Europe 600: 104 groups for 157 features; 80 features sit in 27 groups of near-duplicates; 114 pairs with |rho| >= 0.9


,size,features
0,4,"QTLU20, RSV20, RANK20, RANK30"
1,4,"ROC20, SUMN20, SUMP20, SUMD20"
2,4,"ROC10, SUMN10, SUMP10, SUMD10"
3,4,"ROC30, SUMN30, SUMP30, SUMD30"
4,4,"RANK60, RSV60, MA60, QTLU60"
5,4,"ROC5, SUMN5, SUMP5, SUMD5"
6,4,"ROC60, SUMN60, SUMP60, SUMD60"
7,3,"CNTN60, CNTP60, CNTD60"
8,3,"CNTP30, CNTN30, CNTD30"
9,3,"CNTN20, CNTP20, CNTD20"


## 8. Market volatility

Annualized 20-day realized volatility of the market indexes: the standard deviation of daily index returns (`std_price_change_20_*` in the market file) times $\sqrt{252}$. The US universes share one market file, and so do the EU ones, so their charts repeat. The yearly means are the starting point for the volatility regimes of Step 8.

In [13]:
for u in UNIVERSES:
    s = summaries[u]
    viz.volatility_chart(s["market_volatility"], u).show()
    display(volatility_by_year(s["market_volatility"]).style.format("{:.1%}").set_caption(UNIVERSE_NAMES[u]))

index,DJI.INDX,GSPC.INDX,NDX.INDX
year,,,
2015,14.7%,14.6%,16.9%
2016,11.9%,12.4%,15.3%
2017,6.5%,6.6%,10.0%
2018,15.3%,14.4%,19.8%
2019,12.7%,12.6%,16.5%
2020,28.3%,26.6%,30.5%
2021,12.0%,12.5%,17.2%
2022,19.2%,23.7%,32.1%
2023,11.4%,13.1%,18.3%


index,DJI.INDX,GSPC.INDX,NDX.INDX
year,,,
2015,14.7%,14.6%,16.9%
2016,11.9%,12.4%,15.3%
2017,6.5%,6.6%,10.0%
2018,15.3%,14.4%,19.8%
2019,12.7%,12.6%,16.5%
2020,28.3%,26.6%,30.5%
2021,12.0%,12.5%,17.2%
2022,19.2%,23.7%,32.1%
2023,11.4%,13.1%,18.3%


index,DJI.INDX,GSPC.INDX,NDX.INDX
year,,,
2015,14.7%,14.6%,16.9%
2016,11.9%,12.4%,15.3%
2017,6.5%,6.6%,10.0%
2018,15.3%,14.4%,19.8%
2019,12.7%,12.6%,16.5%
2020,28.3%,26.6%,30.5%
2021,12.0%,12.5%,17.2%
2022,19.2%,23.7%,32.1%
2023,11.4%,13.1%,18.3%


index,SX5E.INDX,SXXP.INDX
year,,
2015,22.2%,18.7%
2016,20.6%,18.2%
2017,9.9%,8.2%
2018,12.9%,11.6%
2019,12.7%,11.3%
2020,27.1%,23.3%
2021,14.3%,12.0%
2022,22.0%,18.1%
2023,13.9%,11.4%


index,SX5E.INDX,SXXP.INDX
year,,
2015,22.2%,18.7%
2016,20.6%,18.2%
2017,9.9%,8.2%
2018,12.9%,11.6%
2019,12.7%,11.3%
2020,27.1%,23.3%
2021,14.3%,12.0%
2022,22.0%,18.1%
2023,13.9%,11.4%


## Observations

All five universes, study period 2015-2024.

1. **Two feature sets, little overlap.** 296 of the 360 Alpha360 columns are momentum-type (past prices over today's close), and the two sets share only momentum, volume and candle features. Alpha158 vs Alpha360 comparisons must use **source** and **horizon**.
2. **Universe size** at the start of each test year: DJI 30 members, NASDAQ-100 101-104, S&P 500 503-505, EURO STOXX 50 50-51, STOXX Europe 600 600-615. The price files also hold past members (DJI 43-44 tickers, STOXX Europe 600 742-749).
3. **The EU membership history starts on 2020-09-30** (EURO STOXX 50 and STOXX Europe 600). At 2020-01-01 no stock passes FinBench's rule "member at the start of the test year", so **the 2020 test year is empty for the EU universes**: their rolling grid is 2021-2024 unless membership is backfilled.
4. **Tickers mapped to the wrong company** (section 5), in test universes:
   - DJI: `PRG.US` is listed as *The Procter & Gamble Company*, but its prices are those of PROG Holdings ($12-64 over 2015-2024, while P&G traded at $51-175). It is one of the 30 stocks in **every** DJI test year. The S&P 500 data has P&G correctly as `PG.US`.
   - STOXX Europe 600: `TEN.US` (*Tenaris*) is Tsakos Energy Navigation, test years 2021-2023; `TOM.F` (*Tomra Systems*) is Toyota Motor, test years 2021-2022 (the correct Tomra listing, `TOM.OL`, is used from 2023).
   - Outside the test universes (harmless): NASDAQ-100 `VBVT.US` (*Viacom*, a penny stock) and `GOLD.US` (*Randgold*), S&P 500 `MNK_old.US` (*Mallinckrodt*, a closed-end fund).
5. **Bad prices that reach a model** (section 5):
   - STOXX Europe 600: `SAABY.US` (Saab B) has a constant price of 992,237 from its first row (2019-03-21) to 2024-05-15, then about 11: five years of zero returns and a −99.999% label in the 2024 test year. `ORRON.ST` (Lundin Energy) falls to 0.0002 with 50 labels of +100% or more in 2019-2022 (test years 2021-2022). Several US OTC listings (`AKBLF.US`, `FCHRF.US`) repeat one price for the whole decade.
   - S&P 500: `DISH.US` is recorded at 0 after its merger with EchoStar, a −100% label in the last week of the 2023 test year. Signature Bank and First Republic failed in 2023 (real), but their post-failure prices stay in the 2023 test year (Signature Bank falls to $0.002, then gains +1,835%). Reused tickers (`MON.US` after Monsanto, `XL.US` after XL Group) repeat a price for years.
   - EURO STOXX 50: `VIV.PA`'s adjusted close is 0.044 instead of 4.11 on 2015-12-31. The extreme labels fall in December 2015, before every rolling window; only the rolling features of early 2016 see the bad price.
6. **The data-quality filter discards them** (decided 2026-09-29; FinBench keeps them): wrong-company tickers, stale prices, prices below 0.01 and one-day jumps beyond ×3. It removes 0.07% (EURO STOXX 50) to 0.9% (STOXX Europe 600) of the samples, and 2.3% for DJI, all of them `PRG.US`. It brings the yearly standard deviation of the label back in line with market volatility (EURO STOXX 50 2015: 73% → 3.9%; STOXX Europe 600 2024: 30% → 4.4%; NASDAQ-100 2022: 17.5% → 7.0%). What it leaves in the rolling windows is mostly real: the crash and rebound of March 2020, Rivian after its IPO, the 2022 sanctions on Russian miners, the collapse of Orpea. Two errors remain in part: `ORRON.ST`'s penny prices above 0.01 (30 labels, test years 2021-2022) and 11 post-failure labels of Signature Bank (2023).
7. **Missing values are negligible**: under 0.8% per feature, except NASDAQ-100 (up to 3%) and STOXX Europe 600 (up to 4.8%), both in CORD, the correlation between price and volume changes. The models fill them with 0, the training median.
8. **Labels.** After the filter, the 5-day return has a median between −0.2% and 0.7% and a 5th-95th percentile range of ±10-12% in 2020, ±4-8% in most other years. The share of positive labels is 48-59%, so guessing "up" is barely better than a coin flip: the weak signal the outline starts from.
9. **Alpha158 is highly redundant**, consistently across universes: the 157 features form 103-105 groups, and 80-82 features sit in 27-29 groups of near-duplicates (|ρ| ≥ 0.9). SUMP/SUMN/SUMD and VSUMP/VSUMN/VSUMD are exact rescalings of each other (SUMN = 1 − SUMP, SUMD = 2 SUMP − 1), so 20 features add no information. Attributions must be reported per group as well as per feature.
10. **Volatility regimes.** 2020 is the stressed year (annualized volatility DJI 28%, S&P 500 27%, EURO STOXX 50 27%, STOXX Europe 600 23%, about twice a calm year), then 2022, which is the peak for NASDAQ-100 (32%). 2017 is the calmest everywhere (DJI 6.5%). Test years 2020 and 2022 are the high-volatility regimes for Step 8.
11. **STOXX Europe 600 mixes calendars and listings.** Its exchanges close on different holidays (the LSE was closed on 2022-01-03 and 2023-01-02 while the continent traded), and some members are priced through US OTC (`.US`) or Frankfurt (`.F`) listings rather than their home exchange: on those days the cross-section is partial.